# Fake Review Detection - Model Training Notebook
This notebook demonstrates the end-to-end Machine Learning & NLP pipeline to classify online product reviews as **Genuine** or **Fake** using TF-IDF feature extraction and Logistic Regression.

## 1. Import Libraries

In [ ]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

## 2. Load and Inspect Dataset

In [ ]:
data_path = '../data/reviews.csv'
df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
df.head()

## 3. Data Preprocessing & Text Cleaning

In [ ]:
def clean_review(text):
    if not isinstance(text, str) or not text:
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|<.*?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)

# Handle missing values and label encoding
df = df.dropna(subset=['text']).copy()
label_map = {'CG': 0, 'OR': 1, 'Fake': 0, 'Genuine': 1, 0: 0, 1: 1}
df['target'] = df['label'].map(label_map)
df['clean_text'] = df['text'].apply(clean_review)
df = df[df['clean_text'].str.strip() != ""].copy()

print("Label Distribution:\n", df['target'].value_counts())
df[['text', 'clean_text', 'target']].head()

## 4. Train/Test Split

In [ ]:
X = df['clean_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train set: {len(X_train)} samples | Test set: {len(X_test)} samples")

## 5. TF-IDF Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Matrix Shape:", X_train_tfidf.shape)

## 6. Train Classifier (Logistic Regression)

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model.fit(X_train_tfidf, y_train)
print("Model training complete!")

## 7. Model Evaluation

In [ ]:
y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary')
rec = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Fake', 'Genuine']))

## 8. Save Artifacts

In [ ]:
os.makedirs('../model', exist_ok=True)
joblib.dump(model, '../model/model.pkl')
joblib.dump(vectorizer, '../model/vectorizer.pkl')
print("Model and Vectorizer saved successfully.")